## Building RAG with LangChain

We will use two key abstractions:
* The LLM
* The Retriever

Langchain objects repond to '.invoke()'

We will also use the vectorstore we created in D2.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

#can also use below to swap out models easily
from langchain_anthropic import ChatAnthropic
from langchain_ollama import ChatOllama

In [2]:
MODEL = 'gpt-4.1-nano'
DB_NAME = 'vector_db'
load_dotenv(override=True)

True

In [3]:
# connect to chroma and use hugging face embedding model (can use OpenAI Embeddings)
# models that embed docs and the query MUST be the same to match dimensions
embeddings = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2')
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Setup LLM and Retriever

In [4]:
# langchain objects
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model=MODEL)

In [5]:
retriever.invoke('Who is Avery?')

[Document(id='fcef9c71-3a1c-4a43-b647-95fdac1f9bda', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [ ]:
# Doesn't return anything - we must now connect the retriever to the LLM so it can provide additional Context
llm.invoke('Who is Avery?')

AIMessage(content='Could you please provide more context or specify which Avery you are referring to? There are many individuals and characters named Avery.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 11, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_8d474bce1e', 'id': 'chatcmpl-DfRFTLTFtFtco26tI6oFrMY2CYGys', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--59584264-014a-4299-b4dd-0bdce519d8ae-0', usage_metadata={'input_tokens': 11, 'output_tokens': 24, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [7]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = '\n\n'.join(doc.page_content for doc in docs) #extract content (see doc JSON Above)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)]) # very similar to chat completions
    return response.content

In [ ]:
answer_question('who is avery?', []) #history = empty list for now

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She has been with the company since its founding in 2015 and has played a key role in establishing Insurellm as a leading player in the insurance technology industry. Avery is known for her innovative leadership, risk management expertise, and her active involvement in professional development, diversity initiatives, and community outreach.'

## Productionising and using GradioUI

See implementation directory.

Modules...

**Ingest.py**
* Read Knowledge Base
* Turn docs into chunks
* Vectorise chunks
* Store in chroma.

**answer.py**
* fetch_context(question) -> retriever
* answer_question(question, history) -> retriver, llm

**app.py** 
* gradio interface

from implementation directory -> uv run ingest.py

from week5 directory -> uv run app.py


In [ ]:
#no history so uses salary for its search so chooses the wrong person as salary for sarah was found first.
# building history -> convo context info
gr.ChatInterface(answer_question).launch()

# responses can be very basic due to further details being left out in other chunks -> losing relevance
# must balance context from history so LLM doesnt struggle when details from previous conversations clog its ability to handle new topics

/Users/alexanderpikelis/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.
